# Pipeline PROMETA: SRA → Trim Galore/FastQC → Salmon → expressão diferencial

A tabela `metadata_geral.csv` continua no formato atual (`sample_id,condition,study_id`), com um SRR por linha. Copie-a para esta pasta ou configure o caminho `METADATA` em `prometa_pipeline.py`. O layout single/paired é detectado pelos FASTQs gerados; o pipeline grava status em `run_status.csv`, metadados enriquecidos em `metadata_processado.csv` e logs/relatórios em `logs/` e `qc/`.

O script de expressão diferencial é `analise_diferencial.R`. Ele exige R com `tximport`, `DESeq2`, `readr` e `dplyr`, além de `reference/tx2gene.csv` com colunas `TXNAME,GENEID`. Cada SRR deve representar uma réplica biológica independente para que a inferência estatística seja válida.


In [ ]:
# Carrega as funções do pipeline. Ajuste os caminhos no arquivo prometa_pipeline.py antes de executar.
from prometa_pipeline import run_pipeline


In [ ]:
# Descarga, conversão, QC/trim e quantificação. Quantificações existentes são puladas.
metadata_processado = run_pipeline()
metadata_processado.head()


In [ ]:
# Análise diferencial por study_id (treatment vs control) com tximport + DESeq2.
from pathlib import Path
import shutil
import subprocess

if shutil.which("Rscript") is None:
    raise EnvironmentError("Rscript não está no PATH. Instale R e os pacotes descritos no início do notebook.")
if not Path("reference/tx2gene.csv").is_file():
    raise FileNotFoundError("Crie reference/tx2gene.csv com colunas TXNAME e GENEID para agregar transcritos a genes.")
subprocess.run(["Rscript", "analise_diferencial.R", str(Path.cwd())], check=True)


## Resumo de consistência entre comparações

A etapa abaixo usa somente resultados do DESeq2 que passam os critérios configurados (`padj <= 0,05` e `|log2FoldChange| >= 1`). O On-Score/On-Ratio é um resumo de direção e recorrência; não substitui os testes estatísticos nem deve ser calculado diretamente a partir de médias de TPM.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

result_dir = Path("differential_expression")
files = sorted(p for p in result_dir.glob("*_DESeq2_treatment_vs_control.csv") if p.name != "DESeq2_todos_estudos.csv")
if not files:
    raise FileNotFoundError("Não há resultados por estudo. Execute primeiro a análise diferencial.")

comparisons = {}
for path in files:
    result = pd.read_csv(path)
    required = {"gene_id", "log2FoldChange", "padj"}
    if not required.issubset(result.columns):
        raise ValueError(f"Colunas ausentes em {path}: {sorted(required - set(result.columns))}")
    study = path.name.replace("_DESeq2_treatment_vs_control.csv", "")
    result = result.dropna(subset=["gene_id", "log2FoldChange", "padj"])
    comparisons[study] = result.set_index("gene_id")

all_genes = sorted(set().union(*(set(df.index) for df in comparisons.values())))
summary = pd.DataFrame(index=pd.Index(all_genes, name="gene_id"))
for study, result in comparisons.items():
    lfc = result["log2FoldChange"].reindex(summary.index)
    padj = result["padj"].reindex(summary.index)
    significant = (padj <= 0.05) & (lfc.abs() >= 1)
    summary[f"Log2FC_{study}"] = lfc
    summary[f"Significant_{study}"] = significant.fillna(False)
    summary[f"Direction_{study}"] = np.select([significant & (lfc >= 1), significant & (lfc <= -1)], [1, -1], default=0)

direction_cols = [c for c in summary.columns if c.startswith("Direction_")]
summary["On_Score_UP"] = (summary[direction_cols] == 1).sum(axis=1)
summary["On_Score_DOWN"] = (summary[direction_cols] == -1).sum(axis=1)
summary["Bono_Net_Score"] = summary["On_Score_UP"] - summary["On_Score_DOWN"]
summary["On_Ratio_UP"] = summary["On_Score_UP"] / len(comparisons)
summary["On_Ratio_DOWN"] = summary["On_Score_DOWN"] / len(comparisons)
summary.to_csv(result_dir / "onscore_onratio_resumo.csv")
summary.sort_values(["On_Score_UP", "On_Score_DOWN"], ascending=False).to_csv(result_dir / "resumo_ordenado.csv")
print(f"Resumo salvo; comparações válidas: {len(comparisons)}; genes: {len(summary)}")
